In [ ]:
import einops 
import timm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import resnet18,  ResNet18_Weights 
from torchvision import transforms

import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

from posetail.datasets.datasets import Rat7mDataset, Rat7mIterableDataset, custom_collate_2d
from train_utils import *

%load_ext autoreload
%autoreload 2

# NOTE: pip install torchvision==0.15.2 --extra-index-url https://download.pytorch.org/whl/cu118


In [2]:
def pad_img(img): 
    
    pad_top = (224 - 177) // 2 
    pad_bottom = 224 - 177 - pad_top 
    pad_left = 0
    pad_right = 0

    padded_img = F.pad(img, 
        (pad_left, pad_right, pad_top, pad_bottom),
        mode = 'reflect' 
        # mode = 'constant', 
        # value = 0
    )

    return padded_img

In [3]:
resnet = resnet18(weights = ResNet18_Weights.IMAGENET1K_V1)
layers = list(resnet.children())
model = nn.Sequential(*layers[:3])

In [ ]:
# testing iterable dataset
run_id = 'run-20250423_201321-kmadlx6h'
config_path = f'/allen/aind/scratch/katie.rupp/wandb/{run_id}/files/config.toml'
config = load_config(config_path)

device = (torch.device('cuda') if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')

video_path = '/allen/aind/scratch/katie.rupp/data/rat7m/videos/s5-d2/s5-d2-camera6-0.mp4'
data_path = '/allen/aind/scratch/katie.rupp/data/rat7m/data/mocap-s5-d2.mat'

# iterable dataset
dataset = get_dataset(**config.dataset.train)

# non-iterable dataset
dataset = Rat7mDataset(
    video_path = video_path, 
    data_path = data_path, 
    n_frames = config.dataset.test.n_frames, 
    max_res = -1)

dataloader = DataLoader(
    dataset, 
    batch_size = config.dataset.batch_size, 
    collate_fn = custom_collate_2d)

for j, batch in enumerate(dataloader):
    views = [view.to(device) for view in batch.views]
    coords = batch.coords.to(device)
    fnum = batch.fnums.to(device)
    break

print(views[0].shape)

torch.Size([1, 16, 1048, 1328, 3])


res: full, layers: 3, downsample_factor: 2, latent_dim: 64
res: full, layers: 4, downsample_factor: 4, latent_dim: 64
res: full, layers: 5, downsample_factor: 4, latent_dim: 64
res: full, layers: 6, downsample_factor: 8, latent_dim: 64

res: 512, layers: 3, downsample_factor: 2, latent_dim: 64
res: 512, layers: 4, downsample_factor: 4, latent_dim: 64
res: 512, layers: 5, downsample_factor: 4, latent_dim: 64

res: 256, layers: 3, downsample_factor: 2, latent_dim: 64


In [59]:
layers[5]

Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (drop_block): Identity()
    (act1): ReLU(inplace=True)
    (aa): Identity()
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act2): ReLU(inplace=True)
    (downsample): Sequential(
      (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (1): BasicBlock(
    (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (drop_block): Identity()
    (act1): ReLU(inplace=True)
    (aa): Identity

In [62]:
n_layers = 7
resnet = timm.create_model('resnet18', features_only = True, pretrained = True)
layers = list(resnet.children())
model = nn.Sequential(*layers[:n_layers])

img = views[0][0, 0, :, :, :]
img = einops.rearrange(img, 'h w c -> c h w')
img = pad_img(img)
img = img / 255.0
print(img.shape)

features = model(img.unsqueeze(0))
features = features.detach().numpy()
print(features.shape)

# plt.imshow(features[0, 30, :, :])

torch.Size([3, 1095, 1328])
(1, 256, 69, 83)


In [11]:
img = views[0][0, 0, :, :, :]
img = einops.rearrange(img, 'h w c -> c h w')
img = pad_img(img)
img = img / 255.0
print(img.shape)

weights = ResNet18_Weights.DEFAULT
# preprocess = weights.transforms()
# img_transformed = preprocess(img)
transform = transforms.Compose([
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225])
])

img_transformed = transform(img)
print(img_transformed.shape)

batch = img_transformed.unsqueeze(0)
features = model(batch)

features = features.detach().numpy()
print(features.shape)

torch.Size([3, 1095, 1328])
torch.Size([3, 1095, 1328])
(1, 64, 548, 664)


In [ ]:
plt.imshow(features[0, 20, :, :])